In [104]:
import pandas as pd
import sys
print(sys.executable)

/workspaces/crypto/.venv/bin/python


In [105]:
df = pd.read_csv('data.csv')

In [106]:
df["month"] = pd.to_datetime(df["Time"]).dt.to_period("M")
df["dia"] = pd.to_datetime(df["Time"]).dt.to_period("D")
df['Amount'] = df['Amount'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['Fee'] = df['Fee'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['Executed'] = df['Executed'].str.replace(r'[A-Za-z]+', '', regex=True).astype(float)
df['profit']=  df['Amount'] - df['Fee']

In [107]:
df_grouped = df.groupby(['Pair', 'Side','month', 'dia']).agg({ 'Executed': 'sum','Amount': 'sum','Fee': 'sum', 'profit': 'sum'}).sort_values(by=['dia', 'Pair']).reset_index(drop=False)
df_grouped.head()

,Pair,Side,month,dia,Executed,Amount,Fee,profit
0,SOLEUR,BUY,2024-01,2024-01-08,2.0,168.0000,0.002000,167.998000
1,COTIUSDT,BUY,2024-01,2024-01-11,1476.0,92.9880,1.476000,91.512000
2,EURUSDT,SELL,2024-01,2024-01-11,1831.0,2005.4943,2.005494,2003.488806
3,FDUSDUSDT,BUY,2024-01,2024-01-11,2006.0,2003.1916,0.000000,2003.191600
4,FDUSDUSDT,SELL,2024-01,2024-01-11,503.0,502.2455,0.000000,502.245500


In [108]:

# Pega o profit só das linhas de BUY, e "arrasta" pra baixo até a próxima compra
df_grouped['ultima_compra'] = df_grouped['profit'].where(df_grouped['Side'] == 'BUY')
df_grouped['ultima_compra'] = df_grouped.groupby('Pair')['ultima_compra'].ffill()
df_grouped.loc[df_grouped['Side'] == 'BUY', 'Amount_anterior'] = 0

# Cria um id que muda a cada nova compra (define os "ciclos" BUY -> SELLs)
df_grouped['buy_group'] = df_grouped.groupby('Pair')['Side'].transform(lambda s: (s == 'BUY').cumsum())

# Soma o profit das vendas, acumulando dentro de cada ciclo (desde a última compra)
df_grouped['sell_profit'] = df_grouped['profit'].where(df_grouped['Side'] == 'SELL', 0)
df_grouped['profit_acumulado'] = df_grouped.groupby(['Pair', 'buy_group'])['sell_profit'].cumsum()

# Lucro real = acumulado das vendas até agora - custo da compra
df_grouped['lucro'] = df_grouped.apply(
    lambda row: row['profit_acumulado'] - row['ultima_compra'] if row['Side'] == 'SELL' else None,
    axis=1
)
df_grouped = df_grouped.sort_values(['Pair', 'dia']).reset_index(drop=True)

# Cria um id que muda a cada nova compra (define os "ciclos" BUY -> SELLs)
df_grouped['buy_group'] = df_grouped.groupby('Pair')['Side'].transform(lambda s: (s == 'BUY').cumsum())

# Executed (quantidade de ETH) só nas linhas de SELL, 0 nas linhas de BUY
df_grouped['sell_qty'] = df_grouped['Executed'].where(df_grouped['Side'] == 'SELL', 0)

# Acumula a quantidade de ETH vendida dentro de cada ciclo (desde a última compra)
df_grouped['qty_eth_acumulada'] = df_grouped.groupby(['Pair', 'buy_group'])['sell_qty'].cumsum()

df_grouped[df_grouped['Pair'] == 'ETHEUR']
df_grouped[df_grouped['Pair'] == 'ETHEUR']

drop=['Fee',  'profit',  'buy_group',  'sell_profit',  'Side_anterior','Amount_anterior']
df_grouped = df_grouped.drop(columns=drop, errors='ignore')

In [109]:
df_grouped[(df_grouped['Pair']=='AAVEUSDC') #& df_grouped['lucro'].notna()
           ]

,Pair,Side,month,dia,Executed,Amount,ultima_compra,profit_acumulado,lucro,sell_qty,qty_eth_acumulada
5,AAVEUSDC,BUY,2025-08,2025-08-22,9.444,2965.41600,2965.406556,0.000000,NaN,0.000,0.000
6,AAVEUSDC,SELL,2025-08,2025-08-25,9.434,3066.05000,2965.406556,3062.983950,97.577394,9.434,9.434
7,AAVEUSDC,BUY,2025-09,2025-09-01,2.575,798.25000,798.247425,0.000000,NaN,0.000,0.000
8,AAVEUSDC,BUY,2025-09,2025-09-13,6.408,2044.15200,2044.145592,0.000000,NaN,0.000,0.000
9,AAVEUSDC,SELL,2025-10,2025-10-10,8.000,1191.50136,2044.145592,1190.369434,-853.776158,8.000,8.000
10,AAVEUSDC,SELL,2026-02,2026-02-06,0.975,111.63750,2044.145592,1302.006807,-742.138785,0.975,8.975


In [110]:
df_grouped['Pair'].unique()

<StringArray>
[  '1000SATSUSDT', '1MBABYDOGEUSDT',       'AAVEUSDC',        'AMPUSDT',
        'ARBUSDT',        'BCHUSDC',        'BIOUSDC',        'BIOUSDT',
        'BLZUSDT',         'BNBEUR',        'BNBUSDC',        'BNBUSDT',
       'BOMEUSDC',       'CITYUSDT',        'CLVUSDT',       'COTIUSDT',
       'CTSIUSDT',       'CTXCUSDT',       'DOGEUSDT',        'DOTUSDC',
        'DOTUSDT',        'EDUUSDT',        'ENAUSDC',        'ENAUSDT',
         'ETHEUR',      'ETHFIUSDT',        'ETHUSDC',        'EURUSDC',
        'EURUSDT',       'FARMUSDT',      'FDUSDUSDT',      'FLOKIUSDC',
      'FLOKIUSDT',       'ICPFDUSD',        'ICPUSDT',         'IDUSDT',
         'IOUSDT',         'IQUSDT',       'JUPFDUSD',        'JUPUSDT',
       'MASKUSDT',      'MATICUSDT',       'MINAUSDT',       'NEARUSDT',
        'NFPUSDT',        'NKNUSDT',       'OMNIUSDT',     'PENDLEUSDT',
     'PEOPLEUSDT',       'PEPEUSDC',       'PEPEUSDT',       'PNUTUSDC',
     'PORTALUSDT',       'PUMPUSDC', 